In [0]:
import matplotlib.pyplot as plt

OUT = "/Volumes/workspace/public_health/outputs/charts"
dbutils.fs.mkdirs(OUT)

caf = spark.sql("""
    WITH who AS (
      SELECT TimeDim AS year, NumericValue AS who
      FROM workspace.public_health.bronze_who
      WHERE IndicatorCode = 'WHOSIS_000001' AND Dim1 = 'SEX_BTSX' AND SpatialDim = 'CAF'
    ),
    wb AS (
      SELECT CAST(date AS INT) AS year, value AS world_bank
      FROM workspace.public_health.bronze_worldbank
      WHERE indicator.id = 'SP.DYN.LE00.IN' AND countryiso3code = 'CAF' AND value IS NOT NULL
    )
    SELECT wb.year, who.who, wb.world_bank
    FROM wb LEFT JOIN who ON wb.year = who.year
    WHERE wb.year BETWEEN 2000 AND 2023
    ORDER BY wb.year
""").toPandas()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(caf["year"], caf["who"], marker="o", label="WHO")
ax.plot(caf["year"], caf["world_bank"], marker="o", label="World Bank (UN WPP)")
ax.set_title("Central African Republic, life expectancy at birth, two sources")
ax.set_xlabel("Year")
ax.set_ylabel("Years")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

path = f"{OUT}/caf_life_expectancy_two_sources.png"
fig.savefig(path, dpi=150)
print("saved", path)
display(fig)

**What it shows.** Life expectancy at birth for Central African Republic, 2000 to 2023, from WHO and from the World Bank (which takes its figure from UN World Population Prospects).

**What it means.** The WHO line moves by fractions of a year, which is how life expectancy behaves. The World Bank line drops to 40.3 in 2014, 31.5 in 2019, 18.8 in 2022, then jumps to 57.4 in 2023. No population lives like that. A rule as simple as "flag any year-on-year change over 3 years" would catch every one of those points. This is the first anomaly the project found, by looking, before any detection code was written, and it is why silver keeps a per-row record of which source each value came from rather than picking one source of truth up front.

In [0]:
spread = spark.sql("""
    WITH who AS (
      SELECT SpatialDim AS iso3, TimeDim AS year, NumericValue AS who
      FROM workspace.public_health.bronze_who
      WHERE IndicatorCode = 'WHOSIS_000001' AND Dim1 = 'SEX_BTSX'
    ),
    wb AS (
      SELECT countryiso3code AS iso3, CAST(date AS INT) AS year, value AS wb
      FROM workspace.public_health.bronze_worldbank
      WHERE indicator.id = 'SP.DYN.LE00.IN' AND value IS NOT NULL
    )
    SELECT who.iso3, who.who - wb.wb AS diff
    FROM who JOIN wb ON who.iso3 = wb.iso3 AND who.year = wb.year
    WHERE who.year = 2019
""").toPandas()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(spread["diff"], bins=40, edgecolor="white")
ax.axvline(0, color="black", linewidth=1)
ax.set_title(f"WHO minus UN life expectancy, 2019, {len(spread)} countries")
ax.set_xlabel("Difference in years (positive = WHO higher)")
ax.set_ylabel("Countries")
ax.grid(alpha=0.3)
fig.tight_layout()

path = f"{OUT}/life_expectancy_spread_2019.png"
fig.savefig(path, dpi=150)
print("saved", path)
print("countries:", len(spread))
print("within 1 year:", (spread["diff"].abs() <= 1).sum())
print("within 2 years:", (spread["diff"].abs() <= 2).sum())
print("more than 5 years apart:", (spread["diff"].abs() > 5).sum())
display(fig)

**What it shows.** For every country with a 2019 life expectancy value in both WHO and the World Bank (185 countries), the difference WHO minus World Bank. The vertical line is zero, perfect agreement.

**What it means.** 107 of 185 countries agree within one year and 139 within two. Only 5 are more than five years apart. The two independent estimates agree for most of the world and disagree badly for a handful, so the reconciliation rule cannot be a global "prefer source X". It has to be per row: when the two agree, either is fine; when they diverge past a threshold, the row is flagged and a human decides.

In [0]:
coverage = spark.sql("""
    SELECT 'WHO' AS source, TimeDim AS year, COUNT(DISTINCT SpatialDim) AS countries
    FROM workspace.public_health.bronze_who
    WHERE IndicatorCode = 'WHOSIS_000001' AND Dim1 = 'SEX_BTSX' AND SpatialDimType = 'COUNTRY'
    GROUP BY TimeDim
    UNION ALL
    SELECT 'World Bank', CAST(date AS INT), COUNT(DISTINCT countryiso3code)
    FROM workspace.public_health.bronze_worldbank
    WHERE indicator.id = 'SP.DYN.LE00.IN' AND value IS NOT NULL
      AND countryiso3code IS NOT NULL AND countryiso3code <> ''
    GROUP BY date
    UNION ALL
    SELECT 'OWID', year, COUNT(DISTINCT code)
    FROM workspace.public_health.bronze_owid_life_expectancy
    WHERE code IS NOT NULL AND code NOT LIKE 'OWID%'
    GROUP BY year
""").toPandas()

fig, ax = plt.subplots(figsize=(9, 4.5))
for source, grp in coverage.groupby("source"):
    grp = grp.sort_values("year")
    ax.plot(grp["year"], grp["countries"], label=source)
ax.set_xlim(1950, 2025)
ax.set_title("Life expectancy: entities with a value, by source and year")
ax.set_xlabel("Year")
ax.set_ylabel("Entities with a value")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

path = f"{OUT}/life_expectancy_coverage_by_source.png"
fig.savefig(path, dpi=150)
print("saved", path)
print(coverage[coverage["year"].isin([2019, 2021, 2022, 2023, 2024])].sort_values(["year", "source"]).to_string(index=False))
display(fig)

**What it shows.** How many entities have a life expectancy value each year, by source. OWID starts in 1950, the World Bank in 1960, WHO in 2000.

**What it means.** Three sources, three definitions of "the world". In 2019: WHO 185 countries, OWID 236 entities, World Bank 260 entities including around 45 regional and income-group aggregates that bronze cannot yet tell apart from countries (the axis says "entities" for that reason). WHO's series stops at 2021 while the World Bank runs to 2024, so for 2022 onward there is only one estimate and no disagreement to detect. Silver's country dimension has to settle which 185 to 260 identities count as a country, and gold's anomaly flags have to know when a value had a second opinion and when it did not.